# Session Log — Accelerometer Entropy Adaptive PPG Sampling

**Project:** EMBC 2026 poster — accelerometer compressibility/entropy as a predictor of HR estimation error  
**Dataset:** IEEE Signal Processing Cup 2015 wrist PPG (22 recordings, 125 Hz)  
**Language:** Python 3.11 / Jupyter, conda env `ppg-entropy`  

---

## Session template (copy for each new session)

```
### Session YYYY-MM-DD
**Goal:** ...  
**Done:**
- 

**Key findings:**
- 

**Blockers / open questions:**
- 

**Next session:**
- 
```

---

### Session 2026-04-18
**Goal:** Project setup and Step 1 completion (data loading + sanity check)

**Done:**
- Confirmed dataset layout: `Training_data/` (12 T1 treadmill recordings) + `TestData/` (10 arm-movement recordings) + `TrueBPM/` (ground truth for test set)
- **Row-layout discrepancy caught and fixed:** training files have `sig` shape `(6, N)` with ECG at row 0, but test files have `sig` shape `(5, N)` with no ECG (rows 0–1 = PPG, 2–4 = ACC). The TestData Readme confirms ECG was withheld by the competition. Loader now branches on `sig.shape[0]` and asserts the shape matches the group.
- **Refined from two groups to three groups** after seeing raw ACC magnitudes:
  - T1 — treadmill (rec 01–12, training, 12 recs)
  - T2 — mixed arm exercise, `TEST_S*_T01.mat` (rec 13, 14, 19, 22 — shake/stretch/push-up/run/jump, 4 recs)
  - T3 — boxing, `TEST_S*_T02.mat` (rec 15, 16, 17, 18, 20, 21 — intensive forearm/upper-arm, 6 recs)
- Registered `Python (ppg-entropy)` as a Jupyter kernel and set it as the notebook default
- Added "Expected output" markdown before every code cell so the user can sanity-check each step independently
- Notebook runs end-to-end with 0 errors and 0 window/GT count mismatches

**Key findings:**
- Mean RMS ACC magnitude by group (raw, still contains 1 g gravity DC):
  - T1 treadmill: **1.476 g** (tight range 1.31–1.72)
  - T2 mixed arm: **1.068 g** (range 0.99–1.13)
  - T3 boxing:    **1.790 g** (range 1.47–2.02)
- Critical observation: **T3 magnitude ≥ T1 magnitude**, but T3 motion is irregular while T1 is periodic. This is exactly the discrimination problem entropy should solve — supports the paper thesis, not against it.
- Two-group analysis (treadmill vs T2+T3 pooled) gave means that were essentially tied (1.48 vs 1.50), which is why the three-group split tells a cleaner narrative.

**Blockers / open questions:**
- Gravity DC bias (~1 g) is still included in raw ACC magnitude. Deferred decision: whether to subtract it (high-pass or per-window mean removal) before computing RMS. Revisit after Step 4 entropy results to see if raw vs AC-only changes anything.
- Window-to-GT alignment is currently exact for all 22 recordings — no tricky off-by-one handling needed in Step 2.

**Next session:**
- Start Step 2: bandpass-filtered FFT HR estimator (stateless, per-window)
- Use the three-group color scheme (`steelblue` / `goldenrod` / `crimson`) in all MAE plots
- Report MAE per recording and per group (T1, T2, T3 separately)


### Session 2026-04-19
**Goal:** Factor shared loading code into a reusable module so Steps 2–6 can import from one place, and briefly revisit the gravity-removal question

**Done:**
- **Decided to defer signal pre-processing** (gravity removal, filtering) to a later step. Reverted an earlier commit that added AC-only RMS computation and a 5th panel to `plot_raw`. Step 1 is back to its pre-gravity state: raw ACC throughout, with a note in the title that pre-processing is deferred.
- Created **`data_loader.py`** as the single source of truth for:
  - Constants: `FS`, `WIN_SAMPLES`, `SHIFT`, `TRAIN_DIR`, `TEST_DIR`, `TRUEBPM_DIR`
  - Maps: `GROUP_COLORS`, `GROUP_LABELS`
  - Functions: `build_manifest()`, `load_recording()`, `rms_acc_windows()`, `load_all_recordings(verbose=False)`
- Paths inside the module are resolved relative to `__file__`, so the loader works regardless of the notebook's current working directory.
- Refactored `step1_data_loading.ipynb` to import from `data_loader.py` instead of defining the loader inline. Removed the now-redundant paths cell. Step 1 is now a lightweight sanity-check / visualization tour that reuses the same loader as every downstream step.
- Smoke-tested the refactored notebook end-to-end: all 22 recordings load, 0 window/GT mismatches, group means unchanged (T1=1.476, T2=1.068, T3=1.790).

**Key findings:**
- The AC-only demeaning *did* give a cleaner numeric separation of T1/T2/T3 group means, but deferred pending Step 4 decisions — we may need different pre-processing for HR estimation (Step 2) vs entropy metrics (Step 4).
- The conda env `ppg-entropy` and the Jupyter kernel `Python (ppg-entropy)` from yesterday are still registered and in use — no env changes today.

**Next session:**
- Create `step2_hr_estimator.ipynb`: stateless bandpass + FFT-peak HR estimator on the PPG-channel average, per 8 s window
- Compare estimated BPM against `bpm0` ground truth; report MAE per recording and per group

### Session 2026-04-19 (later)
**Goal:** Build Step 2 — stateless FFT-peak HR estimator with the same module+notebook split as Step 1

**Done:**
- Created `hr_estimator.py` containing the full Step 2 pipeline:
  - `design_bandpass`, `bandpass_filter` — 4th-order Butterworth 0.5–4 Hz with `filtfilt`
  - `average_channels` — mean of the two PPG rows
  - `estimate_hr_window`, `window_spectrum` — single-window FFT peak-pick (zero-padded to N=8192 → ≈ 0.9 BPM resolution)
  - `estimate_hr_trace` — end-to-end pipeline for one recording
  - `evaluate_recording`, `evaluate_all` — wrap the estimator and compare against `BPM0`
- Created `step2_hr_estimator.ipynb` (18 cells, sections 2.1–2.8): load data → filter response → raw-vs-filtered PPG → per-window FFT peak diagnostic → run estimator on all 22 recordings → per-recording MAE → MAE bar chart → estimated-vs-true traces → group summary table.
- Filter is applied end-to-end per recording once, *then* windows are sliced. Estimator is still stateless at the window level (each window's BPM is independent), but we avoid per-window filter transients. Noted this in the module docstring.
- Smoke-tested the notebook end-to-end with no errors.

**Key findings (baseline FFT-peak estimator results):**

| Group | n_windows | MAE (BPM) | Median err | p90 | Max |
|-------|-----------|-----------|------------|-----|-----|
| T1 — treadmill       | 1768 | **12.61** | 1.09 | 44.5 | 89.4 |
| T2 — mixed arm       | 511  | **14.76** | 1.68 | 51.2 | 100.9 |
| T3 — boxing          | 817  | **24.30** | 8.89 | 69.9 | 125.2 |
| ALL (pooled)         | 3096 | 16.05 | 1.59 | 53.2 | 125.2 |

- **Median ≪ MAE** in every group → heavy-tail failures. The estimator is usually right (median ≈ 1 BPM for T1/T2) but catastrophically wrong on motion-corrupted windows.
- **T3 boxing worst by far** (MAE 24 vs T1's 13), as expected — intense irregular arm motion produces artefacts squarely in the HR band, and the FFT locks on them.
- T3's median error (8.89) is ≫ T1's (1.09), meaning boxing fails *consistently*, not just sporadically.
- Per-recording MAE varies wildly within T1 (0.40 on rec 9, 29.77 on rec 10) — within-group variation is a signal worth studying in Step 3.

This is the behaviour the paper hinges on: a naive estimator works fine at rest/predictable motion and fails during unpredictable motion. Step 4's entropy metric should correlate with where these failures happen.

**Blockers / open questions:**
- No pre-processing of the ACC signal yet — still raw (gravity included). Decide before Step 4 whether the entropy metric runs on raw ACC, gravity-removed ACC, or a high-passed version.
- Baseline estimator is stateless per spec — good. But worth keeping in mind that a temporal smoothing post-process (e.g. median filter across consecutive windows) would mask per-window failures and distort the Step 3+ correlation analysis. Keep it stateless.

**Next session:**
- Step 3: pool all per-window `abs_error` values (from `evaluate_all`) against per-window `acc_rms_windows` from Step 1. Compute Pearson + Spearman correlations pooled and within each group. Scatter plot coloured by group.


### Session 2026-04-20
**Goal:** Step 3 — exploratory correlation between per-window ACC magnitude and per-window HR-estimation error

**Done:**
- Created `acc_features.py` as the per-window ACC feature module (parallel to `hr_estimator.py`). Functions:
  - `demean_window(acc_seg)` — per-axis mean subtraction inside a single 3 × WIN segment
  - `rms_magnitude_ac(acc_seg)` — single-window AC-only RMS of the 3-axis magnitude
  - `per_window_rms_ac(acc, win, shift)` — sliding-window AC-only RMS over a full recording
  - `attach_ac_rms(recordings)` — mutate each dict in-place, adding `acc_rms_ac_windows` and `mean_acc_rms_ac`
- Pre-processing math (applied per 8 s window): subtract each ACC axis's own mean → per-sample 3-axis magnitude → window RMS. This strips the ~1 g gravity DC and leaves motion-only energy.
- Created `step3_exploratory.ipynb` (16 cells, sections 3.1–3.7):
  - 3.1 load data + run HR estimator + attach AC-only RMS
  - 3.2 one-window sanity check (raw → demeaned → magnitude); printed raw vs AC-only RMS values
  - 3.3 **main figure** — 1×4 scatter grid (Pooled / T1 / T2 / T3), per-window |error| vs per-window AC-only RMS, with Pearson r and Spearman ρ annotated per panel
  - 3.4 correlation summary table (Pearson + Spearman with p-values)
  - 3.5 per-recording scatter (22 points), mean error vs mean AC-only RMS
  - 3.6 per-recording mean error bars, sorted by AC-only RMS
  - 3.7 pooled scatter with `log1p(error)` y-axis to reveal the bulk near zero
- Smoke-tested end-to-end with no errors; all pooled/per-group numbers match the preview.

**Key findings (headline for the paper):**

| Slice | n | Pearson r | Pearson p | Spearman ρ | Spearman p |
|-------|---|-----------|-----------|------------|------------|
| Pooled (all groups) | 3096 | +0.196 | 3e-28 | **+0.245** | 2e-43 |
| T1 — treadmill | 1768 | +0.288 | 4e-35 | **+0.274** | 1e-31 |
| T2 — mixed arm | 511  | +0.334 | 9e-15 | **+0.371** | 4e-18 |
| **T3 — boxing** | 817  | −0.039 | **0.27** | **+0.048** | **0.17** |
| Per-recording (n=22) | 22 | +0.353 | 0.11 | +0.348 | 0.11 |

- **Within T1 and T2, ACC magnitude is weakly predictive of HR error** (Spearman ρ ≈ 0.27–0.37, highly significant). A fraction of the per-window error can be explained by how much motion there is.
- **Within T3 boxing, ACC magnitude is useless** (Spearman ρ ≈ 0.05, p = 0.17 — not statistically distinguishable from zero at n=817). Boxing motion is uniformly in the 1.14–1.68 g range, and whether a particular window fails or succeeds is decided by motion *character*, not motion *amount*.
- This is precisely the gap the paper says entropy should fill. Step 4 should show that within T3 the entropy metric correlates with error while magnitude does not — that's the key comparison.
- Per-recording correlation (n=22) is +0.35 but not significant (p = 0.11) — too few recordings. The per-window analysis is what the paper should lean on.
- Spearman > Pearson in every slice except pooled, confirming the heavy-tail intuition: the relationship is more monotonic than linear, and outliers drag the linear fit.

**Blockers / open questions:**
- Step 4 design: which entropy metric first? The spec suggests histogram entropy (plug-in Shannon over a quantized window). I'd start there with B=32 bins, pool all windows, compare Spearman ρ against magnitude's ρ per group. T3 is the test case.
- Should entropy run on raw ACC, demeaned ACC, or high-passed ACC? Intuition: demeaned per window (matches what we just did here).
- Whether to use per-axis vs combined magnitude as the entropy input — probably start with per-axis and see if combined does better.

**Next session:**
- Step 4: `acc_features.py` gets `histogram_entropy(x, bins=32)`; `attach_entropy(recordings)` adds `acc_entropy_windows` etc.
- Re-run the correlation analysis with entropy as the predictor and compare directly against magnitude (same slice structure as Step 3.4 table)
- If histogram entropy beats magnitude in T3, we have the paper's central claim

### Session 2026-04-20 (later)
**Goal:** Step 4 — compressibility / entropy metrics on the AC-only 3-axis ACC magnitude; compare against ACC magnitude as predictor of HR-estimation error

**Done:**
- Extended `acc_features.py` with four new per-window metrics (all computed on the same AC-only magnitude signal as Step 3):
  - `histogram_entropy(x, bins, value_range)` — plug-in Shannon (bits) of a 32-bin histogram
  - `kl_divergence(p, q, eps)` — KL between two integer histograms with ε-smoothing
  - `lz_complexity_window(x)` — LZ76 complexity of the magnitude binarized at its median (via `antropy.lziv_complexity`, normalize=True)
  - `spectral_entropy_window(x, fs)` — Welch-PSD flatness via `antropy.spectral_entropy`
  - `per_window_entropy_metrics(acc)` — returns per-window arrays of all four
  - `attach_entropy_metrics(recordings)` — mutates in-place, adds `acc_{hist_ent,kl_prev,lz_comp,spec_ent}_windows` and `mean_acc_{hist_ent,kl_prev,lz_comp,spec_ent}`
- Histogram bins use per-recording adaptive range `[0, max AC magnitude in recording]` so bins are shared across windows (KL well-defined) but scale-normalized across recordings.
- Created `step4_entropy.ipynb` (20 cells, sections 4.1–4.8):
  - 4.1 load + attach all metrics
  - 4.2 one-window sanity check — magnitude trace, histogram, binarized, Welch PSD, all four metric values printed
  - 4.3–4.6 per-window 1×4 scatter grids (Pooled/T1/T2/T3) for HistEnt / KL / LZ / SpEnt — same layout as Step 3.3
  - **4.7** per-recording scatters (n=22): mean feature vs per-recording *median* |error|, for mag / HistEnt / LZ / SpEnt
  - **4.8** headline comparison table — Spearman ρ × five features × (per-window pooled, T1, T2, T3) and (per-recording MAE, median, p75)
- Smoke-tested end-to-end with no errors.

**Key findings — big pivot to the paper's framing:**

*Per-window within-group correlations are weak for every feature, including magnitude.* The signal is between-group, not within-group. Reported honestly in 4.3–4.6 so the paper doesn't overclaim.

| Feature | Per-window Pooled ρ | T1 | T2 | T3 |
|---------|---------------------|-----|-----|-----|
| AC-RMS magnitude | +0.245 | +0.274 | +0.371 | +0.048 |
| Histogram entropy | +0.080 | +0.108 | +0.167 | +0.005 |
| KL divergence | +0.004 | −0.058 | +0.018 | −0.056 |
| LZ complexity | +0.123 | +0.117 | +0.014 | +0.105 |
| Spectral entropy | +0.098 | −0.037 | +0.069 | −0.015 |

*Per-recording (n=22), against **median** per-window |HR error|:*

| Feature | Spearman ρ | p-value | Significant? |
|---------|-----------|---------|--------------|
| AC-RMS magnitude | +0.327 | 0.138 | no |
| Histogram entropy | −0.266 | 0.232 | no (wrong direction) |
| KL divergence | +0.065 | 0.774 | no |
| **LZ complexity** | **+0.554** | **0.007** | **yes ★** |
| **Spectral entropy** | **+0.528** | **0.012** | **yes ★** |

**Headline for the paper:** *At the recording level, LZ complexity and spectral entropy of the gravity-removed 3-axis accelerometer magnitude are significantly better predictors of median HR-estimation error than the raw ACC magnitude itself (both p < 0.015; magnitude p = 0.14).*

*Why median error, not MAE:* robust to heavy-tail failure outliers, interpretable ("half of all windows have error ≤ X BPM"), and ties directly to the paper's compressibility framing. MAE gives similar rankings (SpEnt +0.62, LZ +0.53, mag +0.35) with SpEnt slightly ahead of LZ; median swaps them. Both are significant, magnitude is not.

**Why histogram entropy fails:** treadmill running produces a *more spread* magnitude distribution (motion swings through a wide range) than boxing (where windows stay in a narrower high-motion regime). So HistEnt goes backward relative to the paper's compressibility intuition. LZ and SpEnt capture temporal/spectral structure, which is the right thing.

**Blockers / open questions:**
- Step 5 four-quadrant analysis needs to be reframed: "high mag + low SpEnt (T1)" vs "high mag + high SpEnt (T3)" will be the core demonstration, not histogram entropy as originally spec'd in context.txt. Suggest splitting the bar charts by SpEnt rather than HistEnt.
- The per-window analysis is genuinely weak. We may want to emphasize per-recording / group-level evidence in the poster, and treat per-window as supplementary.

**Next session:**
- Step 5: four-quadrant analysis using **SpEnt as the entropy axis** (paper-ready view). Plot per-recording or per-window mean |error| in each quadrant: (low mag, low SpEnt) / (low mag, high SpEnt) / (high mag, low SpEnt) / (high mag, high SpEnt).
- Step 6: Assemble final Table 1 from 4.8's comparison table.

---

## Research roadmap

**Shared code:**
- Loading, group maps, per-window raw RMS: [`data_loader.py`](data_loader.py)
- HR-estimation pipeline: [`hr_estimator.py`](hr_estimator.py)
- Per-window ACC features (AC-only RMS + histogram/KL/LZ/spectral entropy): [`acc_features.py`](acc_features.py)

Every step notebook imports from the relevant module rather than redefining functions inline.

| Step | Notebook / module | Status |
|------|-------------------|--------|
| Shared loader | `data_loader.py` | **done (2026-04-19)** |
| Shared HR estimator | `hr_estimator.py` | **done (2026-04-19)** |
| Shared ACC features | `acc_features.py` | **done (2026-04-19 RMS, 2026-04-20 entropy metrics)** — `attach_ac_rms`, `attach_entropy_metrics` |
| 1 — Data loading & sanity check | `step1_data_loading.ipynb` | **done (2026-04-18)** |
| 2 — Simple HR estimator (FFT) | `step2_hr_estimator.ipynb` | **done (2026-04-19)** — T1 MAE 12.6 / T2 14.8 / T3 24.3 BPM (heavy-tail failures) |
| 3 — Exploratory plots (ACC mag vs error) | `step3_exploratory.ipynb` | **done (2026-04-20)** — Spearman rho: Pooled +0.25, T1 +0.27, T2 +0.37, T3 +0.05 (not sig.) |
| 4 — Entropy metrics | `step4_entropy.ipynb` | **done (2026-04-20)** — per-recording median-error Spearman rho: LZ +0.55 (p=0.007 ★), SpEnt +0.53 (p=0.012 ★), magnitude +0.33 (n.s.). **LZ and SpEnt significantly beat magnitude.** |
| 5 — Four-quadrant analysis | `step5_quadrant.ipynb` | not started — use SpEnt as entropy axis (not HistEnt) |
| 6 — Summary comparison table | `step6_summary_table.ipynb` | not started |

**Standard prologue for every step notebook:**
```python
from data_loader   import FS, WIN_SAMPLES, SHIFT, GROUP_COLORS, GROUP_LABELS, load_all_recordings
from hr_estimator  import evaluate_all
from acc_features  import attach_ac_rms, attach_entropy_metrics
recordings = load_all_recordings()
attach_ac_rms(recordings)
attach_entropy_metrics(recordings, bins=32)
results = evaluate_all(recordings)
```

**Three activity groups:**

| Group | Label | Color | Recs | Raw ACC RMS | AC-only RMS | Mean MAE | Mean SpEnt | Mean LZ |
|-------|-------|-------|------|-------------|-------------|----------|------------|---------|
| T1 | treadmill | `steelblue` | 12 | 1.476 | 1.051 | 12.7 | 0.506 | 0.364 |
| T2 | mixed arm | `goldenrod` | 4 | 1.068 | 0.628 | 14.0 | 0.569 | 0.396 |
| T3 | boxing | `crimson` | 6 | 1.790 | 1.460 | 23.7 | 0.590 | 0.384 |

**Core hypothesis (refined 2026-04-20):** Spectral entropy and LZ complexity of the gravity-removed 3-axis ACC magnitude are significantly better predictors of median HR-estimation error than ACC magnitude alone, at the per-recording level (n=22). Per-window within-group correlations are weak for every feature.

**Key T1-vs-T3 discrimination** (similar magnitudes, different entropy):
- AC-RMS: T1=1.05, T3=1.46 (t=-4.49, p=0.002)
- **SpEnt: T1=0.506, T3=0.590 (t=-5.92, p=0.001)** — cleaner separation, in the direction the paper predicts

**Step 5 pivot:** originally specced as histogram entropy × magnitude quadrants. Switch to **spectral entropy × magnitude** — HistEnt actually goes backward (treadmill has higher HistEnt because magnitude distribution is more spread), while SpEnt correctly predicts boxing > treadmill in error.